# 04 — Rete Neurale Siamese di Riferimento (`SiameseV3` — Task Binario Hotspot vs Stabile)

## Scopo del Notebook
Addestrare e validare il modello di riferimento **`SiameseV3`** (`src/models.py`) alimentato dalla **FCGR Baseline ($k=6$, $64 \times 64$)** sul task esclusivamente **binario**:
- **Classe `0`**: Sequenza Stabile / Healthy
- **Classe `1`**: Hotspot di eccDNA / Disease-associated

### Pipeline di Training e Validazione Downstream
1. **Batch Bilanciati per Classe**: ogni batch contiene esattamente il 50% di anchor di classe `0` e il 50% di anchor di classe `1`.
2. **Random Warm-up (`3` epoche)** seguito da **Dynamic Semi-Hard Negative Mining** (fino a `15` epoche con early stopping).
3. **Metriche Pairwise (Coppie Simili vs Dissimili)**: **ROC-AUC**, **PR-AUC**, **F1-Score**, **Precision**, **Recall**, **Accuracy**, $d^+$, $d^-$, **Gap** $\Delta d$ e **$d'$**.
4. **Metriche di Classificazione Binaria Diretta (`0` vs `1`)**: tramite similarità coseno su *Balanced Reference Bank* sul Test Set finale.
5. **Ispezione Visiva FCGR + Distanze di Embedding**: visualizzazione diretta delle FCGR in input e della distanza calcolata dalla rete.

In [ ]:
import copy
import json
import math
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
)

from pathlib import Path
import sys

# Rilevamento automatico Google Colab vs Ambiente Locale
COLAB_ROOT = Path('/content/drive/MyDrive/eccdna_project')
if COLAB_ROOT.exists():
    PROJECT_ROOT = COLAB_ROOT
else:
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'fcgr_analysis'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

META_PATH = RAW_DIR / 'eccdna_disease_detection_metadata.tsv'
FASTA_PATH = RAW_DIR / 'eccdna_disease_detection.body.fa'
MANIFEST_PATH = PROCESSED_DIR / 'siamese_primary_manifest.tsv'

print('PROJECT_ROOT :', PROJECT_ROOT)
print('META_PATH    :', META_PATH, '-> exists:', META_PATH.exists())
print('FASTA_PATH   :', FASTA_PATH, '-> exists:', FASTA_PATH.exists())

from src.fcgr import plot_fcgr_diagnostics, plot_siamese_pair_feedback
from src.dataset import (
    RandomSiamesePairDataset,
    BalancedSiamesePairDataset,
    BalancedSemiHardCandidateDataset,
    SingleFCGRDataset,
)
from src.models import (
    SiameseV3,
    EuclideanContrastiveLoss,
    select_semihard_negatives,
    evaluate_pairwise,
    extract_embeddings,
    compute_topk_similarity_matrix,
)

## 1. Configurazione Iperparametri e Dispositivo GPU

In [ ]:
K = 6
RANDOM_STATE = 42
N_CLASSES = 2  # Task Binario: 0 = Stabile/Healthy, 1 = Hotspot
CLASS_NAMES = {0: 'Stable / Healthy (0)', 1: 'eccDNA Hotspot (1)'}

EMBEDDING_DIM = 128
EUCLIDEAN_MARGIN = 1.25

WARMUP_EPOCHS = 3
MAX_SEMIHARD_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 4
MIN_DELTA = 1e-4

# 64 è divisibile per N_CLASSES=2 (32 anchor classe 0 + 32 anchor classe 1 per batch)
BALANCED_BATCH_SIZE = 64
VAL_BATCH_SIZE = 128
VAL_PAIRS = 10_000
TEST_PAIRS = 10_000
N_NEGATIVE_CANDIDATES = 6

WARMUP_LR = 5e-4
SEMIHARD_LR = 1e-4
WEIGHT_DECAY = 1e-4

MODEL_OUT_DIR = ARTIFACTS_DIR / 'siamese_baseline_k6'
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

FCGR_PATH = PROCESSED_DIR / 'fcgr_cache' / f'fcgr_k{K}.npy'
FCGR_INDEX_PATH = PROCESSED_DIR / 'fcgr_cache' / f'fcgr_k{K}_index.tsv'
VAL_POOL_PATH = PROCESSED_DIR / 'siamese_val_pair_pool.tsv'
TEST_POOL_PATH = PROCESSED_DIR / 'siamese_test_pair_pool.tsv'

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_STATE)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = DEVICE.type == 'cuda'

print('Device       :', DEVICE, f'({torch.cuda.get_device_name(0)})' if torch.cuda.is_available() else '')
print('AMP Enabled  :', AMP_ENABLED)
print('Output Dir   :', MODEL_OUT_DIR)

## 2. Caricamento Dataset Binario e Cache FCGR (`fcgr_k6.npy`)

In [ ]:
metadata = pd.read_csv(MANIFEST_PATH, sep='\t', dtype={'id': str})
metadata['class_id'] = metadata['class_id'].astype(int)

train_metadata = metadata[metadata['split_cluster'] == 'train'].copy().reset_index(drop=True)
val_metadata = pd.read_csv(VAL_POOL_PATH, sep='\t', dtype={'id': str})
val_metadata['class_id'] = val_metadata['class_id'].astype(int)
test_metadata = pd.read_csv(TEST_POOL_PATH, sep='\t', dtype={'id': str})
test_metadata['class_id'] = test_metadata['class_id'].astype(int)

fcgr_memmap = np.load(FCGR_PATH, mmap_mode='r')
fcgr_index = pd.read_csv(FCGR_INDEX_PATH, sep='\t', dtype={'id': str})
id_to_fcgr_row = dict(zip(fcgr_index['id'], fcgr_index['fcgr_row']))

print(f'Train samples : {len(train_metadata):,}')
print(f'Val samples   : {len(val_metadata):,}')
print(f'Test samples  : {len(test_metadata):,}')
print(f'FCGR cache    : {fcgr_memmap.shape} ({fcgr_memmap.dtype})')
display(pd.DataFrame({
    'Train': train_metadata['class_id'].value_counts(),
    'Val': val_metadata['class_id'].value_counts(),
    'Test': test_metadata['class_id'].value_counts(),
}))

## 3. Creazione dei DataLoader Bilanciati e Istanziazione di `SiameseV3`

In [ ]:
WARMUP_BATCHES_PER_EPOCH = math.ceil(len(train_metadata) / BALANCED_BATCH_SIZE)
WARMUP_PAIRS_PER_EPOCH = WARMUP_BATCHES_PER_EPOCH * BALANCED_BATCH_SIZE

MINING_BATCHES_PER_EPOCH = math.ceil((WARMUP_PAIRS_PER_EPOCH / 2) / BALANCED_BATCH_SIZE)
MINING_ANCHORS_PER_EPOCH = MINING_BATCHES_PER_EPOCH * BALANCED_BATCH_SIZE

warmup_train_dataset = BalancedSiamesePairDataset(
    metadata=train_metadata,
    fcgr_memmap=fcgr_memmap,
    id_to_row=id_to_fcgr_row,
    n_batches=WARMUP_BATCHES_PER_EPOCH,
    batch_size=BALANCED_BATCH_SIZE,
    positive_fraction=0.50,
    seed=RANDOM_STATE,
)
mining_dataset = BalancedSemiHardCandidateDataset(
    metadata=train_metadata,
    fcgr_memmap=fcgr_memmap,
    id_to_row=id_to_fcgr_row,
    n_batches=MINING_BATCHES_PER_EPOCH,
    batch_size=BALANCED_BATCH_SIZE,
    n_negative_candidates=N_NEGATIVE_CANDIDATES,
    seed=RANDOM_STATE,
)
val_pair_dataset = RandomSiamesePairDataset(
    metadata=val_metadata,
    fcgr_memmap=fcgr_memmap,
    id_to_row=id_to_fcgr_row,
    n_pairs=VAL_PAIRS,
    positive_fraction=0.50,
    seed=RANDOM_STATE + 50_000,
)
test_pair_dataset = RandomSiamesePairDataset(
    metadata=test_metadata,
    fcgr_memmap=fcgr_memmap,
    id_to_row=id_to_fcgr_row,
    n_pairs=TEST_PAIRS,
    positive_fraction=0.50,
    seed=RANDOM_STATE + 60_000,
)

warmup_train_loader = DataLoader(warmup_train_dataset, batch_size=BALANCED_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=AMP_ENABLED)
mining_loader = DataLoader(mining_dataset, batch_size=BALANCED_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=AMP_ENABLED)
val_pair_loader = DataLoader(val_pair_dataset, batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=AMP_ENABLED)
test_pair_loader = DataLoader(test_pair_dataset, batch_size=VAL_BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=AMP_ENABLED)

model = SiameseV3(embedding_dim=EMBEDDING_DIM, in_channels=1).to(DEVICE)
criterion = EuclideanContrastiveLoss(margin=EUCLIDEAN_MARGIN)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parametri trainabili SiameseV3 : {n_params:,}')
print(f'Warm-up pairs / epoca          : {len(warmup_train_dataset):,}')
print(f'Mining anchors / epoca         : {len(mining_dataset):,} ({2*len(mining_dataset):,} pair constraints)')

## 4. Training: Fase 1 (Random Warm-Up) + Fase 2 (Dynamic Semi-Hard Mining)

In [ ]:
def train_random_epoch(model, loader, dataset, criterion, optimizer, scaler, epoch):
    model.train()
    dataset.set_epoch(epoch)
    total_loss, total_samples = 0.0, 0
    all_targets, all_distances = [], []
    start_time = time.perf_counter()

    for batch in loader:
        x1 = batch['x1'].to(DEVICE, non_blocking=True)
        x2 = batch['x2'].to(DEVICE, non_blocking=True)
        target = batch['target'].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=DEVICE.type, dtype=(torch.float16 if DEVICE.type == 'cuda' else torch.bfloat16), enabled=AMP_ENABLED):
            z1, z2 = model(x1, x2)
        loss, distances = criterion(z1, z2, target)

        if AMP_ENABLED:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        bs = target.shape[0]
        total_loss += loss.item() * bs
        total_samples += bs
        all_targets.append(target.detach().cpu().numpy())
        all_distances.append(distances.detach().cpu().numpy())

    targets = np.concatenate(all_targets)
    distances = np.concatenate(all_distances)
    pos, neg = distances[targets == 1], distances[targets == 0]
    return {
        'loss': total_loss / total_samples,
        'auc': float(roc_auc_score(targets, -distances)),
        'd_pos': float(pos.mean()),
        'd_neg': float(neg.mean()),
        'gap': float(neg.mean() - pos.mean()),
        'seconds': time.perf_counter() - start_time,
    }


def train_dynamic_semihard_epoch(model, loader, dataset, criterion, optimizer, scaler, epoch):
    dataset.set_epoch(epoch)
    total_loss, total_samples = 0.0, 0
    all_targets, all_distances, semihard_fractions = [], [], []
    start_time = time.perf_counter()

    for batch in loader:
        anchor = batch['anchor'].to(DEVICE, non_blocking=True)
        positive = batch['positive'].to(DEVICE, non_blocking=True)
        negatives = batch['negatives'].to(DEVICE, non_blocking=True)

        selected_negative, mining_stats = select_semihard_negatives(
            model=model, anchor=anchor, positive=positive, negatives=negatives,
            margin=EUCLIDEAN_MARGIN, amp_enabled=AMP_ENABLED
        )

        model.train()
        optimizer.zero_grad(set_to_none=True)
        bs = anchor.shape[0]
        train_images = torch.cat([anchor, positive, selected_negative], dim=0)

        with torch.autocast(device_type=DEVICE.type, dtype=(torch.float16 if DEVICE.type == 'cuda' else torch.bfloat16), enabled=AMP_ENABLED):
            embeddings = model.encoder(train_images)

        z_a, z_p, z_n = embeddings[:bs], embeddings[bs:2*bs], embeddings[2*bs:]
        z1 = torch.cat([z_a, z_a], dim=0)
        z2 = torch.cat([z_p, z_n], dim=0)
        targets = torch.cat([torch.ones(bs, device=DEVICE), torch.zeros(bs, device=DEVICE)], dim=0)

        loss, distances = criterion(z1, z2, targets)
        if AMP_ENABLED:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        n_pairs = targets.shape[0]
        total_loss += loss.detach().item() * n_pairs
        total_samples += n_pairs
        all_targets.append(targets.detach().cpu().numpy())
        all_distances.append(distances.detach().cpu().numpy())
        semihard_fractions.append(mining_stats['semihard_fraction'])

    targets_np = np.concatenate(all_targets)
    distances_np = np.concatenate(all_distances)
    pos, neg = distances_np[targets_np == 1], distances_np[targets_np == 0]
    return {
        'loss': total_loss / total_samples,
        'auc': float(roc_auc_score(targets_np, -distances_np)),
        'd_pos': float(pos.mean()),
        'd_neg': float(neg.mean()),
        'gap': float(neg.mean() - pos.mean()),
        'semihard_fraction': float(np.mean(semihard_fractions)),
        'seconds': time.perf_counter() - start_time,
    }

In [ ]:
set_seed(RANDOM_STATE)
model = SiameseV3(embedding_dim=EMBEDDING_DIM, in_channels=1).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=WARMUP_LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)

print('=' * 95)
print('FASE 1: RANDOM-PAIR WARM-UP')
print('=' * 95)
warmup_history = []
for epoch in range(1, WARMUP_EPOCHS + 1):
    tr = train_random_epoch(model, warmup_train_loader, warmup_train_dataset, criterion, optimizer, scaler, epoch)
    va = evaluate_pairwise(model, val_pair_loader, criterion, DEVICE, AMP_ENABLED)
    warmup_history.append({'epoch': epoch, **va, 'train_auc': tr['auc'], 'seconds': tr['seconds']})
    print(f"Warm-up {epoch:02d}/{WARMUP_EPOCHS} | train AUC {tr['auc']:.4f} | val AUC {va['auc']:.4f} | d+ {va['d_pos']:.4f} | d- {va['d_neg']:.4f} | gap {va['gap']:.4f} | d' {va['d_prime']:.4f} | {tr['seconds']:.1f}s")

post_warmup_state = copy.deepcopy(model.state_dict())

print('\n' + '=' * 95)
print('FASE 2: DYNAMIC SEMI-HARD NEGATIVE MINING')
print('=' * 95)
CHECKPOINT_PATH = MODEL_OUT_DIR / 'best_siamese_baseline.pt'
HISTORY_PATH = MODEL_OUT_DIR / 'training_history.tsv'

optimizer = torch.optim.AdamW(model.parameters(), lr=SEMIHARD_LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)

best_metrics = evaluate_pairwise(model, val_pair_loader, criterion, DEVICE, AMP_ENABLED)
best_auc = best_metrics['auc']
best_epoch = 0
epochs_no_improve = 0
history = []

torch.save({'epoch': 0, 'model_state_dict': copy.deepcopy(model.state_dict()), 'val_metrics': best_metrics, 'k': K}, CHECKPOINT_PATH)

for epoch in range(1, MAX_SEMIHARD_EPOCHS + 1):
    tr = train_dynamic_semihard_epoch(model, mining_loader, mining_dataset, criterion, optimizer, scaler, WARMUP_EPOCHS + epoch)
    va = evaluate_pairwise(model, val_pair_loader, criterion, DEVICE, AMP_ENABLED)
    history.append({'epoch': epoch, 'train_loss': tr['loss'], 'train_auc': tr['auc'], 'semihard_fraction': tr['semihard_fraction'], **va, 'seconds': tr['seconds']})

    if va['auc'] > best_auc + MIN_DELTA:
        best_auc = va['auc']
        best_epoch = epoch
        best_metrics = copy.deepcopy(va)
        epochs_no_improve = 0
        torch.save({'epoch': epoch, 'model_state_dict': copy.deepcopy(model.state_dict()), 'val_metrics': best_metrics, 'k': K}, CHECKPOINT_PATH)
        marker = ' <-- BEST'
    else:
        epochs_no_improve += 1
        marker = ''

    print(f"Epoch {epoch:02d}/{MAX_SEMIHARD_EPOCHS} | train AUC {tr['auc']:.4f} | semi-hard {tr['semihard_fraction']*100:.1f}% | val AUC {va['auc']:.4f} | d+ {va['d_pos']:.4f} | d- {va['d_neg']:.4f} | gap {va['gap']:.4f} | d' {va['d_prime']:.4f} | {tr['seconds']:.1f}s{marker}")
    if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
        print('Early stopping attivato.')
        break

pd.DataFrame(history).to_csv(HISTORY_PATH, sep='\t', index=False)

## 5. Validazione Downstream Completa sul Test Set (Metriche Pairwise & Classificazione Binaria)
Carichiamo il miglior checkpoint e calcoliamo tutte le metriche richieste dal progetto:
- **ROC-AUC** e **PR-AUC (Average Precision)**
- **F1-Score, Precision, Recall, Balanced Accuracy**
- **Distribuzione delle distanze di embedding ($d^+$, $d^-$, Gap $\Delta d$, $d'$)**

In [ ]:
best_ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(best_ckpt['model_state_dict'])
model.eval()

@torch.no_grad()
def collect_pair_distances(loader):
    targets_all, dists_all = [], []
    for batch in loader:
        x1 = batch['x1'].to(DEVICE, non_blocking=True)
        x2 = batch['x2'].to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, dtype=(torch.float16 if DEVICE.type == 'cuda' else torch.bfloat16), enabled=AMP_ENABLED):
            z1, z2 = model(x1, x2)
        dists = torch.linalg.vector_norm(z1.float() - z2.float(), ord=2, dim=1)
        targets_all.append(batch['target'].numpy())
        dists_all.append(dists.cpu().numpy())
    return np.concatenate(targets_all), np.concatenate(dists_all)

val_targets_p, val_dists_p = collect_pair_distances(val_pair_loader)
test_targets_p, test_dists_p = collect_pair_distances(test_pair_loader)

# Ricerca soglia ottimale di distanza su Validation Set per massimizzare F1
thresholds = np.linspace(val_dists_p.min(), val_dists_p.max(), 400)
best_thr = thresholds[int(np.argmax([f1_score(val_targets_p, (val_dists_p <= t).astype(int)) for t in thresholds]))]

test_preds_p = (test_dists_p <= best_thr).astype(int)
test_scores_p = -test_dists_p  # Più vicini = score positivo maggiore

pos_d = test_dists_p[test_targets_p == 1]
neg_d = test_dists_p[test_targets_p == 0]
gap_d = float(neg_d.mean() - pos_d.mean())
d_prime_test = float(gap_d / np.sqrt(0.5 * (pos_d.var() + neg_d.var()) + 1e-12))

pairwise_test_metrics = {
    'pairwise_roc_auc': float(roc_auc_score(test_targets_p, test_scores_p)),
    'pairwise_pr_auc': float(average_precision_score(test_targets_p, test_scores_p)),
    'pairwise_f1': float(f1_score(test_targets_p, test_preds_p)),
    'pairwise_macro_f1': float(f1_score(test_targets_p, test_preds_p, average='macro')),
    'pairwise_precision': float(precision_score(test_targets_p, test_preds_p)),
    'pairwise_recall': float(recall_score(test_targets_p, test_preds_p)),
    'pairwise_accuracy': float(accuracy_score(test_targets_p, test_preds_p)),
    'd_pos_mean': float(pos_d.mean()),
    'd_neg_mean': float(neg_d.mean()),
    'distance_gap': gap_d,
    'd_prime': d_prime_test,
    'threshold': float(best_thr),
}

print('=== METRICHE PAIRWISE SUL TEST SET ===')
display(pd.DataFrame([pairwise_test_metrics]))

In [ ]:
# Classificazione Binaria Diretta (0 = Stabile vs 1 = Hotspot) tramite Reference Bank degli Embedding
train_emb_loader = DataLoader(SingleFCGRDataset(train_metadata, fcgr_memmap, id_to_fcgr_row), batch_size=256, shuffle=False)
test_emb_loader = DataLoader(SingleFCGRDataset(test_metadata, fcgr_memmap, id_to_fcgr_row), batch_size=256, shuffle=False)

train_embeddings, train_labels = extract_embeddings(model, train_emb_loader, DEVICE, AMP_ENABLED)
test_embeddings, test_labels = extract_embeddings(model, test_emb_loader, DEVICE, AMP_ENABLED)

# Costruiamo una Reference Bank bilanciata (5000 per classe o il minimo disponibile)
ref_per_class = min(5000, int((train_labels == 0).sum()), int((train_labels == 1).sum()))
rng = np.random.default_rng(RANDOM_STATE)
ref_idx = np.concatenate([
    rng.choice(np.where(train_labels == 0)[0], size=ref_per_class, replace=False),
    rng.choice(np.where(train_labels == 1)[0], size=ref_per_class, replace=False),
])

sim_matrix = compute_topk_similarity_matrix(
    query_embeddings=test_embeddings,
    reference_embeddings=train_embeddings[ref_idx],
    reference_labels=train_labels[ref_idx],
    n_classes=2,
    top_k=20,
)

# Score continuo per la classe 1 (Hotspot): differenza di similarità rispetto alla classe 0
hotspot_score = sim_matrix[:, 1] - sim_matrix[:, 0]
test_pred_cls = np.argmax(sim_matrix, axis=1)

binary_cls_metrics = {
    'cls_roc_auc': float(roc_auc_score(test_labels, hotspot_score)),
    'cls_pr_auc': float(average_precision_score(test_labels, hotspot_score)),
    'cls_f1': float(f1_score(test_labels, test_pred_cls)),
    'cls_macro_f1': float(f1_score(test_labels, test_pred_cls, average='macro')),
    'cls_precision': float(precision_score(test_labels, test_pred_cls)),
    'cls_recall': float(recall_score(test_labels, test_pred_cls)),
    'cls_balanced_accuracy': float(balanced_accuracy_score(test_labels, test_pred_cls)),
}

print('=== METRICHE CLASSIFICAZIONE BINARIA (HOTSPOT vs STABILE) — TEST SET ===')
display(pd.DataFrame([binary_cls_metrics]))
print('\nClassification Report:')
print(classification_report(test_labels, test_pred_cls, target_names=['Stable (0)', 'Hotspot (1)'], digits=4))

# Salvataggio di tutte le metriche per il Notebook 16
all_summary = {**pairwise_test_metrics, **binary_cls_metrics, 'k': K, 'train_samples': len(train_metadata), 'best_epoch': best_epoch}
pd.DataFrame([all_summary]).to_csv(MODEL_OUT_DIR / 'baseline_final_metrics.csv', index=False)
with open(MODEL_OUT_DIR / 'baseline_final_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(all_summary, f, indent=4)

In [ ]:
# Grafici di Validazione: Distribuzione Distanze di Embedding, Curva ROC e Curva Precision-Recall
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].hist(pos_d, bins=45, alpha=0.65, density=True, label=f'Coppie Positive (d+={pos_d.mean():.3f})')
axes[0].hist(neg_d, bins=45, alpha=0.65, density=True, label=f'Coppie Negative (d-={neg_d.mean():.3f})')
axes[0].axvline(best_thr, linestyle='--', color='black', label=f'Soglia={best_thr:.3f}')
axes[0].set_xlabel('Distanza Euclidea Embedding')
axes[0].set_ylabel('Densità')
axes[0].set_title(f"Distribuzione Distanze (d'={d_prime_test:.3f})")
axes[0].legend()
axes[0].grid(alpha=0.3)

fpr, tpr, _ = roc_curve(test_targets_p, test_scores_p)
axes[1].plot(fpr, tpr, linewidth=2, label=f"Pairwise ROC-AUC = {pairwise_test_metrics['pairwise_roc_auc']:.4f}")
fpr_c, tpr_c, _ = roc_curve(test_labels, hotspot_score)
axes[1].plot(fpr_c, tpr_c, linewidth=2, label=f"Classif. ROC-AUC = {binary_cls_metrics['cls_roc_auc']:.4f}")
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Curve ROC sul Test Set')
axes[1].legend()
axes[1].grid(alpha=0.3)

prec, rec, _ = precision_recall_curve(test_targets_p, test_scores_p)
axes[2].plot(rec, prec, linewidth=2, label=f"Pairwise PR-AUC = {pairwise_test_metrics['pairwise_pr_auc']:.4f}")
prec_c, rec_c, _ = precision_recall_curve(test_labels, hotspot_score)
axes[2].plot(rec_c, prec_c, linewidth=2, label=f"Classif. PR-AUC = {binary_cls_metrics['cls_pr_auc']:.4f}")
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Curve Precision-Recall sul Test Set')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(MODEL_OUT_DIR / 'baseline_test_curves_and_distances.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Feedback Visivo: Ispezione FCGR in Input e Distanze Predette dalla Siamese
Visualizziamo una coppia positiva e una negativa dal Test Set con la distanza Euclidea calcolata da `SiameseV3`.

In [ ]:
test_batch = next(iter(test_pair_loader))
with torch.no_grad():
    z1_b, z2_b = model(test_batch['x1'].to(DEVICE), test_batch['x2'].to(DEVICE))
    _, dists_b = criterion(z1_b, z2_b, test_batch['target'].to(DEVICE))

pos_idx = int((test_batch['target'] == 1).nonzero(as_tuple=True)[0][0].item())
neg_idx = int((test_batch['target'] == 0).nonzero(as_tuple=True)[0][0].item())

for idx, desc in [(pos_idx, 'Coppia Positiva'), (neg_idx, 'Coppia Negativa')]:
    plot_siamese_pair_feedback(
        x1=test_batch['x1'][idx],
        x2=test_batch['x2'][idx],
        target=test_batch['target'][idx].item(),
        label1=f'{desc} (Ramo 1)',
        label2=f'{desc} (Ramo 2)',
        embedding_distance=float(dists_b[idx].item()),
    )